# INCLUDE50 Sign Language Recognition — Training Pipeline
Generates keypoints with mediapipe 0.10.x and trains a transformer model.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────
!pip install mediapipe==0.10.31 transformers timm joblib tqdm -q

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Clone / copy project files ────────────────────────────────────
import os

!git clone https://github.com/BishalDubey27/Major_Project.git

# Move into the INCLUDE subfolder
os.chdir('/content/Major_Project/INCLUDE')
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── 4. Set paths ──────────────────────────────────────────────────────
# Path to the INCLUDE50 dataset videos on your Drive
DATASET_DIR = '/content/drive/MyDrive/INCLUDE_Dataset'  # <-- change this

# Where to save generated keypoints
KEYPOINTS_DIR = '/content/keypoints'
os.makedirs(KEYPOINTS_DIR, exist_ok=True)

# Where to save the trained model
SAVE_PATH = '/content/drive/MyDrive/trained_model'
os.makedirs(SAVE_PATH, exist_ok=True)

print('Dataset dir:', DATASET_DIR)
print('Keypoints dir:', KEYPOINTS_DIR)
print('Save path:', SAVE_PATH)

In [ ]:
# ── 5. Generate keypoints ─────────────────────────────────────────────
# This extracts mediapipe landmarks from all INCLUDE50 videos
!python generate_keypoints.py \
    --include_dir {DATASET_DIR} \
    --save_dir {KEYPOINTS_DIR} \
    --dataset include50

# Check output
import glob
for split in ['train', 'val', 'test']:
    files = glob.glob(f'{KEYPOINTS_DIR}/include50_{split}_keypoints/*.json')
    print(f'include50_{split}_keypoints: {len(files)} files')

In [ ]:
# ── 6. Train the transformer model ───────────────────────────────────
!python runner.py \
    --dataset include50 \
    --model transformer \
    --transformer_size small \
    --data_dir {KEYPOINTS_DIR} \
    --save_path {SAVE_PATH} \
    --epochs 50 \
    --batch_size 128 \
    --learning_rate 1e-4 \
    --use_augs

In [ ]:
# ── 7. Check saved model ──────────────────────────────────────────────
import torch, glob

pth_files = glob.glob(f'{SAVE_PATH}/*.pth')
print('Saved models:', pth_files)

if pth_files:
    cp = torch.load(pth_files[0], map_location='cpu', weights_only=False)
    print('Score:', cp.get('score'))
    for k, v in cp['model'].items():
        if 'l2' in k:
            print('Output layer:', k, v.shape)

In [ ]:
# ── 8. Quick accuracy test on test set ───────────────────────────────
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, '/content/Major_Project/INCLUDE')
from models.transformer import Transformer
from configs import TransformerConfig
from dataset import KeypointsDataset

with open('label_maps/label_map_include50.json') as f:
    label_map = json.load(f)
idx_to_label = {v: k for k, v in label_map.items()}

cp = torch.load(pth_files[0], map_location='cpu', weights_only=False)
config = TransformerConfig(size='small')
model = Transformer(config=config, n_classes=50)
model.load_state_dict(cp['model'])
model.eval()

ds = KeypointsDataset(
    keypoints_dir=f'{KEYPOINTS_DIR}/include50_test_keypoints',
    use_augs=False, label_map=label_map, mode='test', max_frame_len=200
)

correct = 0
for i in range(len(ds)):
    sample = ds[i]
    with torch.no_grad():
        probs = torch.softmax(model(sample['data'].unsqueeze(0)), dim=-1)
        pred = probs.argmax(dim=-1).item()
    if idx_to_label[pred] == sample['lablel_string']:
        correct += 1

print(f'Test accuracy: {correct}/{len(ds)} = {correct/len(ds)*100:.2f}%')

In [ ]:
# ── 9. Download the model ─────────────────────────────────────────────
# The model is already saved to your Drive at SAVE_PATH
# Download it and place it in INCLUDE/ folder of your local project
# Then update unified_app.py model_path to point to the new file

from google.colab import files
files.download(pth_files[0])
print('Downloaded:', pth_files[0])
print()
print('Next step: place this .pth file in your INCLUDE/ folder')
print('Then update unified_app.py model_path to use it')